# Communication pathway probe (dim_hidden = 25)

Trained **two-module RNN** checkpoints only: **`common_input=False`**, **`common_readout=True`**, **`nb_steps=2`**.  
Compare **`input_routing`** = `shared` vs `task_routed` across paper sparsities and a small init-scale grid.

## What this notebook does

1. Resolves run folders via `build_run_id` + `experiments.json`.
2. Loads **`state_*.pt`** + **`sim_*.npz`** for a **small matched participant set** per `(input_routing, init_scale)` (intersection across reference sparsities, default `no_comms` and `1.0`).
3. Forward pass **matching training time depth**: `inputs` are expanded with **`temporal_data(..., nb_steps from settings, noise_ratio=None)`** (same as `schedule.py`), then **`return_core_comms=True`** reports **L2 norms** of `core_final` / `comms_final`, their ratio, and **cosine similarity** between the two halves of `comms_final` (per-module blocks).
4. **Ablation A (readout):** recomputes readout from **`core` pathway sequence only** (drops the **`comms` sequence** before readout) → `mean_l2_delta_readout_core_only`.
5. **Ablation B (Phase B):** temporarily zeros **`comms_mask`** (same storage as `comms.weight_hh_l0` `Masked_weight.mask`) so **`comms` recurrent `weight_hh` is zero** while **`weight_ih` stays** → `mean_l2_delta_zero_comms_weight_hh`. This targets **inter-module recurrent** edges defined at init, not the input→comms path.

## How to read results

- **Readout-only ablation** large, **Phase B (`weight_hh` via mask)** small → comms affects logits mainly via **summed sequence / readout**, not **learned recurrent inter-module `weight_hh`** on this slice.
- **Phase B** large for dense sparsity and ~0 for `no_comms` (already no `weight_hh` support) → recurrent weights matter where trained non-zero.
- If **`ratio_comms_over_core`** is flat, rely on the two **L2 delta** columns.

**Grid note:** `INIT_SCALES` defaults to **`[0.001, 0.01, 0.1]`** because those have **global-init** (`init_scope` global) counterparts in `experiments.json` for dim_hidden=25. If you add e.g. `1.0` or `10`, ensure matching conditions exist (many high-init rows are `input_only` or missing `sp0.5` combinations).

---

## Implementation plan (communication effect → conclusions)

**Goal:** Move from coarse probes to evidence that **recurrent cross-module coupling** (vs whole-comms-subnet / readout mixing) matters, with statistics that match **training**.

| Phase | What | Why | Status |
|-------|------|-----|--------|
| **A. Temporal alignment** | Build inputs with `a1b2.data.temporal.temporal_data` using **`nb_steps` from `settings.json`** (same as `schedule.py`: `noise_ratio=None`). | Training runs **two recurrent steps per trial** when `nb_steps=2`; `seq_len=1` was **under-powering** recurrence (comms included). | **Implemented** in this notebook (`npz_batch` + `nb_steps` column). |
| **B. Targeted ablation** | Zero **`community.comms_mask`** in-place during one forward (restored after); `Masked_weight` on **`weight_hh_l0`** shares this buffer. **`weight_ih`** unchanged. | **Recurrent inter-module comms** vs **input→comms** vs readout-splice ablation. | **Implemented** (`mean_l2_delta_zero_comms_weight_hh`). |
| **C. Behavioral hook** | Same batch: compute **task-relevant** ablation delta (probe 0 → logits 0–1, probe 1 → 2–3) and **accuracy** full vs ablated using saved `labels` in `npz`. | Links internal change to **functional** effect. | **Implemented** (`mean_l2_delta_task_logits`, `acc_full`, `acc_ablate_readout`, `acc_delta_ablate_minus_full`). |
| **D. Population / robustness** | Optional flags: **N participants**, **K trial offsets** or bootstrap; split by **phase** and **probe**. | Avoid “one batch / one seed” conclusions. | *Next:* outer loops + aggregate mean ± SEM in a second table. |
| **E. Saved trajectories** | Compare **`hiddens_post_phase_*_comms_per_module`** vs core for **cross-module** energy from stored runs. | Geometry **without** re-forward assumptions. | *Next:* optional cell reading npz trajectory keys. |

**How to argue “communication matters” after A–E:**  
(1) **B** shows logits or behavior change when **inter-module recurrence** is removed, not only when the whole comms branch is removed.  
(2) Effect **scales with trained sparsity** (dense vs `no_comms`) on the same init/routing.  
(3) **C** shows a matched functional effect on task-relevant outputs/accuracy, not only latent-state movement.  
(4) **D** shows it is not a single-participant artifact.


In [1]:
import os
import sys
import json
from pathlib import Path

import math
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

try:
    from IPython.display import display
except ImportError:
    display = print

from tqdm.auto import tqdm

# --- project root (parent of a1b2 package) ---
project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root containing a1b2/")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.utils.run_config import build_run_id
from a1b2.analysis.run_loader import (
    load_settings,
    build_wrapper_from_settings,
    load_wrapper_state,
    list_participants_with_state,
)

# --- grid ---
DIM_HIDDEN = 25
INIT_SCALES = [0.001, 0.01, 0.1, 1, 2]
SPARSITY_LABELS = ["no_comms", "0.5", "1.0"]  # paper-style
ROUTINGS = ["shared", "task_routed"]

PAPER_SPARSITY_TO_FLOAT = {"no_comms": 0.0, "0.5": 0.5, "1.0": 1.0}

sim_folder = project_root / "data" / "simulations"
config_path = project_root / "a1b2" / "models" / "experiments.json"
settings_all = json.loads(config_path.read_text())

device = torch.device("cpu")

# batch from stored trials (phase 0 = A1)
PHASE = 0
BATCH_SIZE = 64
TRIAL_OFFSET = 0

# matched-participant policy (runtime-safe default)
MAX_MATCHED_PARTICIPANTS_PER_REGIME = 300
MATCH_REFERENCE_SPARSITIES = ["no_comms", "1.0"]
SIMILARITY_ORDER = ["same", "near", "far"]

print("project_root:", project_root)
print("sim_folder:", sim_folder)
print("device:", device)


project_root: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular
sim_folder: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations
device: cpu


/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _init_scope_ok(c):
    s = c.get("init_scope", c.get("init_policy", "global"))
    return s == "global"


def matches_paper_row(c, routing: str, sp_label: str, init_scale: float) -> bool:
    if c.get("arch") != "two_module_rnn":
        return False
    if c.get("dim_hidden") != DIM_HIDDEN:
        return False
    if c.get("nb_steps", 1) != 2:
        return False
    if c.get("common_readout", True) is not True:
        return False
    if c.get("common_input", False) is not False:
        return False
    if c.get("input_routing", "shared") != routing:
        return False
    c_init = float(c.get("init_scale", -1.0))
    if not math.isclose(c_init, float(init_scale), rel_tol=0.0, abs_tol=1e-9):
        return False
    if not _init_scope_ok(c):
        return False
    target_sp = PAPER_SPARSITY_TO_FLOAT[sp_label]
    c_sp = float(c.get("sparsity", 1.0))
    if sp_label == "no_comms":
        if c_sp != 0.0 and "no_comms" not in c.get("name", ""):
            return False
    else:
        if not (abs(c_sp - target_sp) < 1e-6):
            return False
    # exclude init_scope / naming experiments we do not run in the paper grid
    if "input_only" in c.get("name", "") or "initscope" in c.get("name", ""):
        return False
    return True


def find_condition(routing: str, sp_label: str, init_scale: float):
    hits = [
        c
        for c in settings_all["conditions"]
        if matches_paper_row(c, routing, sp_label, init_scale)
    ]
    return hits[0] if hits else None


def similarity_regime_from_participant(participant: str) -> str:
    p = participant.lower()
    if "_same_" in p:
        return "same"
    if "_near_" in p:
        return "near"
    if "_far_" in p:
        return "far"
    return "unknown"


def npz_batch(sim_dir: Path, participant: str, phase: int, offset: int, batch: int, nb_steps: int):
    """Match training (`schedule.py`) and return labels for probe-aligned task metrics."""
    from a1b2.data.temporal import temporal_data

    path = sim_dir / f"sim_{participant}.npz"
    d = np.load(path, allow_pickle=True)
    inp = d["inputs"][phase, offset : offset + batch].astype(np.float32)
    pr = d["probes"][phase, offset : offset + batch].astype(np.float32)
    lab = d["labels"][phase, offset : offset + batch].astype(np.float32)

    input_t = torch.from_numpy(inp)  # (batch, 12)
    if input_t.shape[0] < 2:
        raise ValueError("batch too small")
    if nb_steps > 1:
        x, _ = temporal_data(input_t, nb_steps=nb_steps, noise_ratio=None)
    else:
        x = input_t.unsqueeze(0)  # (1, batch, 12)

    probe = torch.from_numpy(pr)
    labels = torch.from_numpy(lab)  # (batch, 2)
    return x, probe, labels


def collapse_wrapper_logits(wrapper, outputs):
    if isinstance(outputs, list):
        out = torch.stack(outputs, dim=0).sum(dim=0)
        out = out[-1]
    else:
        out = outputs[-1]
    if out.dim() == 2 and out.shape[-1] > wrapper.output_size:
        out = out[:, : wrapper.output_size] + out[:, wrapper.output_size :]
    elif out.dim() == 3 and out.shape[1] == wrapper.n_modules:
        out = out.sum(dim=1)
    return out


def expand_for_wrapper(wrapper, x, feature_probe=None):
    if x.dim() == 2:
        x = x.unsqueeze(0)
    if wrapper.input_routing == "task_routed" and feature_probe is not None:
        return wrapper._routed_input(x, feature_probe)
    return x.repeat(1, 1, wrapper.n_modules)


@torch.no_grad()
def forward_ablate_comms_readout(wrapper, x, feature_probe=None):
    # Test A: remove the entire comms contribution before readout (core-only sequence).
    x_exp = expand_for_wrapper(wrapper, x, feature_probe)
    core_out = wrapper.community.core(x_exp)
    sequence = core_out[0]
    outputs = wrapper.community.readout(sequence)
    return collapse_wrapper_logits(wrapper, outputs)


@torch.no_grad()
def forward_ablate_comms_hh_mask(wrapper, x, feature_probe=None):
    # Test B: keep comms stream present, but zero its HH communication mask during this forward.
    x_exp = expand_for_wrapper(wrapper, x, feature_probe)
    comms_mask = wrapper.community.comms_mask
    backup = comms_mask.clone()
    try:
        comms_mask.zero_()
        outputs, _all_states = wrapper.community(x_exp)
        return collapse_wrapper_logits(wrapper, outputs)
    finally:
        comms_mask.copy_(backup)


@torch.no_grad()
def measure_run(sim_dir: Path, participant: str):
    json_path = sim_dir / "settings.json"
    if not json_path.is_file():
        return None
    st = load_settings(sim_dir)
    nb_steps = int(st.get("condition", {}).get("nb_steps", 1))

    wrapper = build_wrapper_from_settings(st, device=device)
    state_path = sim_dir / f"state_{participant}.pt"
    if not state_path.is_file():
        return None
    load_wrapper_state(wrapper, state_path)
    wrapper.eval()

    x, probe, labels = npz_batch(sim_dir, participant, PHASE, TRIAL_OFFSET, BATCH_SIZE, nb_steps=nb_steps)
    x, probe, labels = x.to(device), probe.to(device), labels.to(device)
    feat = probe if wrapper.input_routing == "task_routed" else None

    out_full, _hid, _tr, core_f, comms_f = wrapper.forward(
        x, feature_probe=feat, return_core_comms=True
    )
    out_ab_readout = forward_ablate_comms_readout(wrapper, x, feature_probe=feat)
    out_ab_mask = forward_ablate_comms_hh_mask(wrapper, x, feature_probe=feat)

    h = wrapper.hidden_size
    nc = core_f.float().norm(dim=1)
    nm = comms_f.float().norm(dim=1)
    c0, c1 = comms_f[:, :h], comms_f[:, h:]
    cos_blocks = F.cosine_similarity(c0, c1, dim=1).mean().item()

    idx_task0 = (probe.reshape(-1).long() == 0)
    idx_task1 = ~idx_task0

    def select_task_logits(out):
        task = torch.zeros((out.shape[0], 2), device=out.device)
        task[idx_task0] = out[idx_task0, 0:2]
        task[idx_task1] = out[idx_task1, 2:4]
        return task

    full_task = select_task_logits(out_full)
    ab_readout_task = select_task_logits(out_ab_readout)
    ab_mask_task = select_task_logits(out_ab_mask)

    def accuracy_from_task_logits(task_logits):
        pred = torch.atan2(task_logits[:, 0], task_logits[:, 1])
        y = torch.atan2(labels[:, 0], labels[:, 1])
        err = torch.atan2(torch.sin(pred - y), torch.cos(pred - y)).abs()
        thr = (np.pi / 8.0)
        return (err <= thr).float().mean().item()

    delta_readout = (out_full - out_ab_readout).float()
    delta_mask = (out_full - out_ab_mask).float()
    task_delta_readout = (full_task - ab_readout_task).float()
    task_delta_mask = (full_task - ab_mask_task).float()

    acc_full = accuracy_from_task_logits(full_task)
    acc_ab_readout = accuracy_from_task_logits(ab_readout_task)
    acc_ab_mask = accuracy_from_task_logits(ab_mask_task)

    row = {
        "nb_steps": nb_steps,
        "seq_len": int(x.shape[0]),
        "norm_core_mean": nc.mean().item(),
        "norm_comms_mean": nm.mean().item(),
        "ratio_comms_over_core": (nm / (nc + 1e-8)).mean().item(),
        "cosine_comms_half_blocks": cos_blocks,
        "mean_l2_delta_logits_readout": delta_readout.pow(2).sum(dim=1).sqrt().mean().item(),
        "max_l2_delta_logits_readout": delta_readout.pow(2).sum(dim=1).sqrt().max().item(),
        "mean_l2_delta_task_logits_readout": task_delta_readout.pow(2).sum(dim=1).sqrt().mean().item(),
        "max_l2_delta_task_logits_readout": task_delta_readout.pow(2).sum(dim=1).sqrt().max().item(),
        "acc_full": acc_full,
        "acc_ablate_readout": acc_ab_readout,
        "acc_delta_readout_minus_full": acc_ab_readout - acc_full,
        "mean_l2_delta_logits_mask": delta_mask.pow(2).sum(dim=1).sqrt().mean().item(),
        "max_l2_delta_logits_mask": delta_mask.pow(2).sum(dim=1).sqrt().max().item(),
        "mean_l2_delta_task_logits_mask": task_delta_mask.pow(2).sum(dim=1).sqrt().mean().item(),
        "max_l2_delta_task_logits_mask": task_delta_mask.pow(2).sum(dim=1).sqrt().max().item(),
        "acc_ablate_mask": acc_ab_mask,
        "acc_delta_mask_minus_full": acc_ab_mask - acc_full,
    }
    return row


In [3]:
rows = []
missing = []

# 1) Resolve available condition folders and participant sets
cond_info = {}
for routing, sp_label, init_scale in product(ROUTINGS, SPARSITY_LABELS, INIT_SCALES):
    key = (routing, sp_label, init_scale)
    cond = find_condition(routing, sp_label, init_scale)
    if cond is None:
        missing.append((routing, sp_label, init_scale, "no_condition"))
        continue
    rid = build_run_id(cond)
    sim_dir = sim_folder / rid
    if not sim_dir.is_dir():
        missing.append((routing, sp_label, init_scale, f"missing_dir:{rid}"))
        continue
    parts = list_participants_with_state(sim_dir)
    if not parts:
        missing.append((routing, sp_label, init_scale, f"no_state:{rid}"))
        continue
    cond_info[key] = {
        "run_id": rid,
        "sim_dir": sim_dir,
        "participants": sorted(parts),
    }

# 2) Build matched participant jobs per (routing, init, similarity_regime)
jobs = []
for routing, init_scale in product(ROUTINGS, INIT_SCALES):
    available_sps = [sp for sp in SPARSITY_LABELS if (routing, sp, init_scale) in cond_info]
    if not available_sps:
        continue

    ref_sps = [sp for sp in MATCH_REFERENCE_SPARSITIES if sp in available_sps]
    if len(ref_sps) == len(MATCH_REFERENCE_SPARSITIES):
        match_sps = ref_sps
    else:
        match_sps = available_sps

    for regime in SIMILARITY_ORDER:
        participant_sets = [
            {
                p
                for p in cond_info[(routing, sp, init_scale)]["participants"]
                if similarity_regime_from_participant(p) == regime
            }
            for sp in match_sps
        ]
        matched = sorted(set.intersection(*participant_sets)) if participant_sets else []
        if not matched:
            missing.append((routing, "|".join(match_sps), init_scale, f"no_matched_participants:{regime}"))
            continue

        selected = matched[:MAX_MATCHED_PARTICIPANTS_PER_REGIME]
        for sp_label in available_sps:
            run_meta = cond_info[(routing, sp_label, init_scale)]
            for participant in selected:
                if participant in run_meta["participants"]:
                    jobs.append((
                        routing,
                        sp_label,
                        init_scale,
                        regime,
                        run_meta["run_id"],
                        run_meta["sim_dir"],
                        participant,
                        len(selected),
                        len(matched),
                        ",".join(match_sps),
                    ))

# 3) Execute jobs with progress bar
_pbar = tqdm(jobs, desc="comms pathway probe", unit="job", total=len(jobs), miniters=1)
for routing, sp_label, init_scale, regime, rid, sim_dir, participant, n_selected, n_matched_total, matched_on in _pbar:
    _pbar.set_postfix_str(f"{routing} · {regime} · {sp_label} · init={init_scale} · p={participant}", refresh=False)
    m = measure_run(sim_dir, participant)
    if m is None:
        missing.append((routing, sp_label, init_scale, f"measure_failed:{participant}"))
        continue
    rows.append(
        {
            "input_routing": routing,
            "similarity_regime": regime,
            "sparsity_label": sp_label,
            "init_scale": init_scale,
            "run_id": rid,
            "participant": participant,
            "n_selected_participants": n_selected,
            "n_matched_participants_total": n_matched_total,
            "matched_on_sparsities": matched_on,
            **m,
        }
    )
_pbar.close()

df = pd.DataFrame(rows)
if not df.empty:
    df["similarity_regime"] = pd.Categorical(df["similarity_regime"], categories=SIMILARITY_ORDER, ordered=True)
    df = df.sort_values(["input_routing", "init_scale", "similarity_regime", "participant", "sparsity_label"]).reset_index(drop=True)

display(df)

if not df.empty:
    metric_cols = [
        "norm_core_mean",
        "norm_comms_mean",
        "ratio_comms_over_core",
        "cosine_comms_half_blocks",
        "mean_l2_delta_logits_readout",
        "max_l2_delta_logits_readout",
        "mean_l2_delta_task_logits_readout",
        "max_l2_delta_task_logits_readout",
        "acc_full",
        "acc_ablate_readout",
        "acc_delta_readout_minus_full",
        "mean_l2_delta_logits_mask",
        "max_l2_delta_logits_mask",
        "mean_l2_delta_task_logits_mask",
        "max_l2_delta_task_logits_mask",
        "acc_ablate_mask",
        "acc_delta_mask_minus_full",
    ]
    agg = (
        df.groupby(["input_routing", "init_scale", "similarity_regime", "sparsity_label"], observed=True)[metric_cols]
        .agg(["mean", "sem", "count"])
        .reset_index()
    )
    display(agg)

if missing:
    print("Skipped / missing:")
    for t in missing:
        print(" ", t)


comms pathway probe:   0%|          | 0/5911 [00:00<?, ?job/s]

comms pathway probe: 100%|██████████| 5911/5911 [01:24<00:00, 69.94job/s, task_routed · far · 1.0 · init=2 · p=study2_far_sub9]            


,input_routing,similarity_regime,sparsity_label,init_scale,run_id,participant,n_selected_participants,n_matched_participants_total,matched_on_sparsities,nb_steps,...,max_l2_delta_task_logits_readout,acc_full,acc_ablate_readout,acc_delta_readout_minus_full,mean_l2_delta_logits_mask,max_l2_delta_logits_mask,mean_l2_delta_task_logits_mask,max_l2_delta_task_logits_mask,acc_ablate_mask,acc_delta_mask_minus_full
0,shared,same,1.0,0.001,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,study1_same_sub1,103,103,"no_comms,1.0",2,...,0.608606,1.000000,1.000000,0.000000,0.117812,0.140466,0.083079,0.106295,1.000000,0.000000
1,shared,same,no_comms,0.001,two_module_rnn_25_no_comms_nb2_init0.001_nb2_s...,study1_same_sub1,103,103,"no_comms,1.0",2,...,0.528834,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
2,shared,same,1.0,0.001,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,study1_same_sub10,103,103,"no_comms,1.0",2,...,0.599762,1.000000,1.000000,0.000000,0.140494,0.144773,0.099065,0.110047,1.000000,0.000000
3,shared,same,no_comms,0.001,two_module_rnn_25_no_comms_nb2_init0.001_nb2_s...,study1_same_sub10,103,103,"no_comms,1.0",2,...,0.518514,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
4,shared,same,1.0,0.001,two_module_rnn_25_nb2_init0.001_nb2_shared_sp1...,study1_same_sub11,103,103,"no_comms,1.0",2,...,0.626858,1.000000,1.000000,0.000000,0.121725,0.131967,0.085827,0.100482,1.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5906,task_routed,far,no_comms,2.000,two_module_rnn_25_task_routed_no_comms_nb2_ini...,study2_far_sub64,101,101,"no_comms,1.0",2,...,0.789436,0.843750,0.578125,-0.265625,0.000000,0.000000,0.000000,0.000000,0.843750,0.000000
5907,task_routed,far,1.0,2.000,two_module_rnn_25_task_routed_nb2_init2_nb2_ta...,study2_far_sub8,101,101,"no_comms,1.0",2,...,1.166254,0.500000,0.421875,-0.078125,0.661772,1.325685,0.427799,1.230435,0.656250,0.156250
5908,task_routed,far,no_comms,2.000,two_module_rnn_25_task_routed_no_comms_nb2_ini...,study2_far_sub8,101,101,"no_comms,1.0",2,...,0.524490,0.578125,0.500000,-0.078125,0.000000,0.000000,0.000000,0.000000,0.578125,0.000000
5909,task_routed,far,1.0,2.000,two_module_rnn_25_task_routed_nb2_init2_nb2_ta...,study2_far_sub9,101,101,"no_comms,1.0",2,...,1.393924,0.843750,0.359375,-0.484375,0.788665,1.186890,0.638777,1.162853,0.265625,-0.578125


input_routing init_scale similarity_regime sparsity_label norm_core_mean  \
                                                                       mean   
0         shared      0.001              same            0.5       0.444350   
1         shared      0.001              same            1.0       0.441645   
2         shared      0.001              same       no_comms       0.447985   
3         shared      0.001              near            0.5       0.444679   
4         shared      0.001              near            1.0       0.441871   
..           ...        ...               ...            ...            ...   
67   task_routed      2.000              near            1.0       2.335948   
68   task_routed      2.000              near       no_comms       2.336929   
69   task_routed      2.000               far            0.5       2.348529   
70   task_routed      2.000               far            1.0       2.327637   
71   task_routed      2.000               far       no_comms       2.329804   

                   norm_comms_mean                  ...  \
         sem count            mean       sem count  ...   
0   0.000892    15        0.493299  0.001460    15  ...   
1   0.000349   103        0.536504  0.000621   103  ...   
2   0.000369   103        0.447985  0.000369   103  ...   
3   0.000891    16        0.495140  0.001541    16  ...   
4   0.000338   101        0.537653  0.000556   101  ...   
..       ...   ...             ...       ...   ...  ...   
67  0.007947   101        2.589756  0.006651   101  ...   
68  0.008023   101        1.532293  0.003808   101  ...   
69  0.030632    10        2.199719  0.015821    10  ...   
70  0.008652   101        2.579088  0.007346   101  ...   
71  0.008579   101        1.530293  0.003988   101  ...   

   mean_l2_delta_task_logits_mask max_l2_delta_task_logits_mask            \
                            count                          mean       sem   
0                              15                      0.060335  0.000969   
1                             103                      0.113601  0.000583   
2                             103                      0.000000  0.000000   
3                              16                      0.063141  0.001000   
4                             101                      0.114600  0.000509   
..                            ...                           ...       ...   
67                            101                      0.943756  0.018570   
68                            101                      0.000000  0.000000   
69                             10                      0.678617  0.054937   
70                            101                      0.937157  0.020676   
71                            101                      0.000000  0.000000   

         acc_ablate_mask                 acc_delta_mask_minus_full            \
   count            mean       sem count                      mean       sem   
0     15        1.000000  0.000000    15                  0.000000  0.000000   
1    103        1.000000  0.000000   103                  0.000000  0.000000   
2    103        1.000000  0.000000   103                  0.000000  0.000000   
3     16        0.861328  0.014677    16                  0.032227  0.010813   
4    101        0.833540  0.007542   101                  0.038676  0.005370   
..   ...             ...       ...   ...                       ...       ...   
67   101        0.417698  0.015536   101                 -0.304610  0.015888   
68   101        0.695854  0.010452   101                  0.000000  0.000000   
69    10        0.467187  0.046959    10                 -0.198437  0.053676   
70   101        0.436262  0.013845   101                 -0.304610  0.015020   
71   101        0.718131  0.009370   101                  0.000000  0.000000   

          
   count  
0     15  
1    103  
2    103  
3     16  
4    101  
..   ...  
67   101  
68   101  
69    10  
70   101  
71   101  

[72 rows x 55 column

Skipped / missing:
  ('shared', 'no_comms', 1, 'no_condition')
  ('shared', '0.5', 1, 'no_condition')
  ('shared', '1.0', 1, 'no_condition')
  ('task_routed', 'no_comms', 1, 'no_condition')
  ('task_routed', '0.5', 1, 'no_condition')
  ('task_routed', '1.0', 1, 'no_condition')


In [4]:
# Post-processing: direct sparsity contrasts (1.0 - no_comms) for BOTH tests.
# IMPORTANT: contrasts are kept separate by similarity regime and are not averaged across regimes.

if "df" not in globals() or df.empty:
    print("No rows in df; run the main analysis cell first.")
else:
    contrast_metrics = [
        "mean_l2_delta_task_logits_readout",
        "acc_delta_readout_minus_full",
        "mean_l2_delta_task_logits_mask",
        "acc_delta_mask_minus_full",
    ]

    wide = (
        df[
            [
                "input_routing",
                "init_scale",
                "similarity_regime",
                "participant",
                "sparsity_label",
                *contrast_metrics,
            ]
        ]
        .pivot_table(
            index=["input_routing", "init_scale", "similarity_regime", "participant"],
            columns="sparsity_label",
            values=contrast_metrics,
            observed=True,
        )
    )

    needed_cols = [
        ("mean_l2_delta_task_logits_readout", "1.0"),
        ("mean_l2_delta_task_logits_readout", "no_comms"),
        ("acc_delta_readout_minus_full", "1.0"),
        ("acc_delta_readout_minus_full", "no_comms"),
        ("mean_l2_delta_task_logits_mask", "1.0"),
        ("mean_l2_delta_task_logits_mask", "no_comms"),
        ("acc_delta_mask_minus_full", "1.0"),
        ("acc_delta_mask_minus_full", "no_comms"),
    ]
    missing_needed = [c for c in needed_cols if c not in wide.columns]
    if missing_needed:
        print("Cannot form requested contrasts; missing columns:", missing_needed)
    else:
        idx = wide.index
        contrast = pd.DataFrame(
            {
                "input_routing": idx.get_level_values("input_routing"),
                "init_scale": idx.get_level_values("init_scale").astype(float),
                "similarity_regime": idx.get_level_values("similarity_regime"),
                "participant": idx.get_level_values("participant"),
                "delta_task_logits_readout_sp1_minus_nocomms": (
                    wide[("mean_l2_delta_task_logits_readout", "1.0")]
                    - wide[("mean_l2_delta_task_logits_readout", "no_comms")]
                ).values,
                "delta_accdelta_readout_sp1_minus_nocomms": (
                    wide[("acc_delta_readout_minus_full", "1.0")]
                    - wide[("acc_delta_readout_minus_full", "no_comms")]
                ).values,
                "delta_task_logits_mask_sp1_minus_nocomms": (
                    wide[("mean_l2_delta_task_logits_mask", "1.0")]
                    - wide[("mean_l2_delta_task_logits_mask", "no_comms")]
                ).values,
                "delta_accdelta_mask_sp1_minus_nocomms": (
                    wide[("acc_delta_mask_minus_full", "1.0")]
                    - wide[("acc_delta_mask_minus_full", "no_comms")]
                ).values,
            }
        )

        contrast["similarity_regime"] = pd.Categorical(
            contrast["similarity_regime"], categories=SIMILARITY_ORDER, ordered=True
        )
        contrast["delta_between_tests_task_logits"] = (
            contrast["delta_task_logits_readout_sp1_minus_nocomms"]
            - contrast["delta_task_logits_mask_sp1_minus_nocomms"]
        )
        contrast["delta_between_tests_accdelta"] = (
            contrast["delta_accdelta_readout_sp1_minus_nocomms"]
            - contrast["delta_accdelta_mask_sp1_minus_nocomms"]
        )
        contrast = contrast.sort_values(
            ["input_routing", "init_scale", "similarity_regime", "participant"]
        ).reset_index(drop=True)
        display(contrast)

        contrast_summary = (
            contrast.groupby(["input_routing", "init_scale", "similarity_regime"], observed=True)
            .agg(
                delta_task_logits_readout_mean=("delta_task_logits_readout_sp1_minus_nocomms", "mean"),
                delta_task_logits_readout_sem=("delta_task_logits_readout_sp1_minus_nocomms", "sem"),
                delta_task_logits_readout_count=("delta_task_logits_readout_sp1_minus_nocomms", "count"),
                delta_accdelta_readout_mean=("delta_accdelta_readout_sp1_minus_nocomms", "mean"),
                delta_accdelta_readout_sem=("delta_accdelta_readout_sp1_minus_nocomms", "sem"),
                delta_accdelta_readout_count=("delta_accdelta_readout_sp1_minus_nocomms", "count"),
                delta_task_logits_mask_mean=("delta_task_logits_mask_sp1_minus_nocomms", "mean"),
                delta_task_logits_mask_sem=("delta_task_logits_mask_sp1_minus_nocomms", "sem"),
                delta_task_logits_mask_count=("delta_task_logits_mask_sp1_minus_nocomms", "count"),
                delta_accdelta_mask_mean=("delta_accdelta_mask_sp1_minus_nocomms", "mean"),
                delta_accdelta_mask_sem=("delta_accdelta_mask_sp1_minus_nocomms", "sem"),
                delta_accdelta_mask_count=("delta_accdelta_mask_sp1_minus_nocomms", "count"),
                delta_between_tests_task_logits_mean=("delta_between_tests_task_logits", "mean"),
                delta_between_tests_task_logits_sem=("delta_between_tests_task_logits", "sem"),
                delta_between_tests_task_logits_count=("delta_between_tests_task_logits", "count"),
                delta_between_tests_accdelta_mean=("delta_between_tests_accdelta", "mean"),
                delta_between_tests_accdelta_sem=("delta_between_tests_accdelta", "sem"),
                delta_between_tests_accdelta_count=("delta_between_tests_accdelta", "count"),
            )
            .reset_index()
            .sort_values(["input_routing", "init_scale", "similarity_regime"])
            .reset_index(drop=True)
        )
        display(contrast_summary)


,input_routing,init_scale,similarity_regime,participant,delta_task_logits_readout_sp1_minus_nocomms,delta_accdelta_readout_sp1_minus_nocomms,delta_task_logits_mask_sp1_minus_nocomms,delta_accdelta_mask_sp1_minus_nocomms,delta_between_tests_task_logits,delta_between_tests_accdelta
0,shared,0.001,same,study1_same_sub1,0.064577,0.000000,0.083079,0.000000,-0.018502,0.000000
1,shared,0.001,same,study1_same_sub10,0.073514,0.000000,0.099065,0.000000,-0.025550,0.000000
2,shared,0.001,same,study1_same_sub11,0.065709,0.000000,0.085827,0.000000,-0.020118,0.000000
3,shared,0.001,same,study1_same_sub12,0.068028,0.000000,0.091642,0.000000,-0.023613,0.000000
4,shared,0.001,same,study1_same_sub13,0.073386,0.000000,0.098615,0.000000,-0.025230,0.000000
...,...,...,...,...,...,...,...,...,...,...
2316,task_routed,2.000,far,study2_far_sub6,0.316712,-0.421875,0.559268,-0.078125,-0.242556,-0.343750
2317,task_routed,2.000,far,study2_far_sub62,0.297521,-0.250000,0.493972,-0.156250,-0.196450,-0.093750
2318,task_routed,2.000,far,study2_far_sub64,0.281333,-0.171875,0.499412,-0.453125,-0.218079,0.281250
2319,task_routed,2.000,far,study2_far_sub8,0.219998,0.000000,0.427799,0.156250,-0.207801,-0.156250


,input_routing,init_scale,similarity_regime,delta_task_logits_readout_mean,delta_task_logits_readout_sem,delta_task_logits_readout_count,delta_accdelta_readout_mean,delta_accdelta_readout_sem,delta_accdelta_readout_count,delta_task_logits_mask_mean,...,delta_task_logits_mask_count,delta_accdelta_mask_mean,delta_accdelta_mask_sem,delta_accdelta_mask_count,delta_between_tests_task_logits_mean,delta_between_tests_task_logits_sem,delta_between_tests_task_logits_count,delta_between_tests_accdelta_mean,delta_between_tests_accdelta_sem,delta_between_tests_accdelta_count
0,shared,0.001,same,7.130647e-02,2.684810e-04,103,0.000000,0.000000,103,9.505163e-02,...,103,0.000000,0.000000,103,-2.374516e-02,1.736127e-04,103,0.000000,0.000000,103
1,shared,0.001,near,7.250476e-02,2.423435e-04,101,0.052444,0.007327,101,9.670033e-02,...,101,0.038676,0.005370,101,-2.419557e-02,1.457988e-04,101,0.013769,0.006339,101
2,shared,0.001,far,6.328745e-02,3.078645e-03,101,0.013459,0.007236,101,5.763183e-02,...,101,0.003403,0.001683,101,5.655622e-03,2.440432e-03,101,0.010056,0.007291,101
3,shared,0.010,same,7.128122e-02,2.713772e-04,103,0.000000,0.000000,103,9.508451e-02,...,103,0.000000,0.000000,103,-2.380328e-02,1.688827e-04,103,0.000000,0.000000,103
4,shared,0.010,near,7.246423e-02,2.432976e-04,101,0.048577,0.007189,101,9.672297e-02,...,101,0.035582,0.005105,101,-2.425874e-02,1.418101e-04,101,0.012995,0.006422,101
5,shared,0.010,far,2.279333e-02,3.704052e-04,101,0.010520,0.007873,101,4.946269e-02,...,101,0.005724,0.002485,101,-2.666936e-02,2.881732e-04,101,0.004796,0.007351,101
6,shared,0.100,same,7.127384e-02,2.716757e-04,103,0.000000,0.000000,103,9.508682e-02,...,103,0.000000,0.000000,103,-2.381299e-02,1.634237e-04,103,0.000000,0.000000,103
7,shared,0.100,near,7.238985e-02,2.473454e-04,101,0.030167,0.006874,101,9.655381e-02,...,101,0.029703,0.004815,101,-2.416396e-02,1.382944e-04,101,0.000464,0.004840,101
8,shared,0.100,far,4.373819e-02,2.164287e-04,101,-0.003403,0.001683,101,5.963997e-02,...,101,-0.000774,0.000774,101,-1.590178e-02,1.521843e-04,101,-0.002630,0.001867,101
9,shared,2.000,same,2.242711e-01,7.875441e-03,103,-0.200546,0.017198,103,5.709098e-01,...,103,-0.398362,0.015364,103,-3.466387e-01,6.804872e-03,103,0.197816,0.018156,103
